In [ ]:
from pulao.logging import init_logging
import logging
from time import sleep
from typing import Tuple

import pandas as pd

from IPython.display import display

from pulao.constant import  FractalType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#

# region 2. Plotly - Cbar
#
df_pandas2 = cbar_manager.df_cbar.to_pandas()
df_pandas2["datetime"] = pd.date_range("2025-01-01", periods=df_pandas2.shape[0], freq="h")
df_pandas2["open_price"] = df_pandas2["high_price"]
df_pandas2["close_price"] = df_pandas2["low_price"]

fig2 = make_subplots(
    rows=1, cols=1
)
fig2.add_trace(go.Candlestick(
    x=df_pandas2['datetime'],
    open=df_pandas2['open_price'],
    high=df_pandas2['high_price'],
    low=df_pandas2['low_price'],
    close=df_pandas2['close_price'],
    name='K线',
), row=1, col=1)

df_swing_point_low2 = df_pandas2[(df_pandas2['fractal_type'] == FractalType.BOTTOM)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_low2['datetime'],
    y=df_swing_point_low2['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_swing_point_low2["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_high2 = df_pandas2[(df_pandas2['fractal_type'] == FractalType.TOP)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_high2['datetime'],
    y=df_swing_point_high2['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_swing_point_high2["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


# title = 'CBar Chart - K线合并处理'
title = 'CBar Chart - 分形重叠处理'
# title = 'CBar Chart - 次级别处理'
# title = 'CBar Chart - 连续同级别同向高低点处理'
fig2.update_layout(title=title,
    height=900,
    hovermode='x unified',    # X轴悬停联动虚线
)

fig2.update_xaxes(
    showgrid=False,
)

fig2.update_yaxes(
    showgrid=False,
)
# fig2 = None
fig2.show()


# endregion

In [1]:
from pulao.logging import init_logging
import logging
from IPython.display import display

from pulao.constant import FractalType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
# trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#
display(swing_manager.df_swing)

cbar_manager.df_cbar

# swing_manager.df_swing

{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "debug", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 05:01:42"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "debug", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 05:01:42"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "debug", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 05:01:42"}
{"fractal": "Fractal(left=CBar(index=0, start_index=0, end_index=1, high_price=942.2349853515625, low_price=935.9240112304688, fractal_type=0), middle=CBar(index=1, start_index=2, end_index=2, high_price=939.052001953125, low_price=930.2979736328125, fractal_type=-1), right=CBar(index=2, start_index=3, end_index=3, high_price=951.2730102539062, low_price=937.4929809570312, fractal_type=0))", "df_swing_height": 1, "event": "情况2，最后3根bar能组成分形。第一次构建，直接添加。", "level": "debug", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 05:01:42"}
{"active_swing": "Swing(index=0, direction=1, start_

index,start_index,end_index,high_price,low_price,direction,is_completed
u32,u32,u32,f32,f32,i8,bool
0,1,7,961.794983,930.297974,1,true
1,7,13,980.258972,947.198975,-1,true
2,13,32,988.528015,948.01001,1,true
3,32,42,965.278015,921.375,-1,true
4,42,72,955.924988,906.143005,1,false


index,start_index,end_index,high_price,low_price,fractal_type
u32,u32,u32,f32,f32,i8
0,0,1,942.234985,935.924011,0
1,2,2,939.052002,930.297974,-1
2,3,3,951.27301,937.492981,0
3,4,5,955.56897,948.888,1
4,6,6,950.607971,946.338989,-1
…,…,…,…,…,…
68,94,94,943.085999,933.172974,0
69,95,95,935.369995,928.525024,-1
70,96,96,935.497986,931.234985,0


In [6]:
from pulao.logging import logger
logger.error("error")

{"event": "error", "level": "error", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 05:03:47"}
